# DepthWizard2 — TEMPORARY DAv2-Small bridge (Colab GPU + Cloudflare quick tunnel)

**Temporary.** The tunnel URL dies when this notebook disconnects, the runtime is recycled, or the session ends. Keep this tab open and connected. Every restart gives a **new** URL, which must be set again as `DAV2_INFERENCE_URL` on the backend. No Cloudflare account or card is involved (quick tunnel). See `docs/DEPLOY.md` in the repo.

Runtime → Change runtime type → **T4 GPU** (or better), then Runtime → Run all.

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print('torch', torch.__version__)

In [ ]:
# transformers pinned to the version the project runs (DPTImageProcessorPil); Colab's own torch/CUDA is kept
!pip -q install transformers==5.17.0 python-multipart fastapi uvicorn

In [ ]:
%%writefile dav2_server.py
"""TEMPORARY Colab bridge: DAv2-Small /predict behind a Cloudflare quick tunnel (docs/DEPLOY.md).

Same response as space/app.py: raw 518x518 relative depth in the compact "u16-zlib" encoding. Model loading and
preprocessing are copied verbatim from scripts/bench/cpu_inference_bench.py; the only
addition is moving the model/input to the GPU when Colab gives one.

  POST /predict  multipart field "file" (PNG/JPG/TIFF RGB) -> JSON (518x518 relative depth, u16-zlib)
  GET  /health   -> {"ok": true, "device": ..., "model": ...}
"""
import base64
import io
import time
import zlib

import numpy as np
import torch
from fastapi import FastAPI, File, HTTPException, UploadFile
from PIL import Image
from transformers import AutoModelForDepthEstimation, DPTImageProcessorPil

mid = "depth-anything/Depth-Anything-V2-Small-hf"
dev = "cuda" if torch.cuda.is_available() else "cpu"

# --- verbatim from scripts/bench/cpu_inference_bench.py -------------------------
proc = DPTImageProcessorPil.from_pretrained(mid); model = AutoModelForDepthEstimation.from_pretrained(mid, use_safetensors=True).eval()
# ---------------------------------------------------------------------------------


def encode_depth(depth) -> dict:
    """Compact wire format "u16-zlib" (docs/DEPLOY.md): min-max quantised to uint16
    (max error (max-min)/131070, ~3e-5 on real outputs), little-endian, zlib, base64.
    ~0.68 MB instead of 1.43 MB for raw float32 at 518x518. Decode:
    q = frombuffer(zlib.decompress(b64decode(data_b64)), "<u2"); depth = min + q/65535*(max-min)."""
    lo, hi = float(depth.min()), float(depth.max())
    q = np.zeros(depth.shape, "<u2") if hi <= lo else np.round((depth - lo) / (hi - lo) * 65535).astype("<u2")
    return {"encoding": "u16-zlib", "shape": list(depth.shape), "min": lo, "max": hi,
            "data_b64": base64.b64encode(zlib.compress(q.tobytes(), 6)).decode("ascii")}
model = model.to(dev)

app = FastAPI(title="DepthWizard2 DAv2-Small bridge (temporary)")


@app.get("/health")
def health() -> dict:
    return {"ok": True, "device": dev, "model": mid}


@app.post("/predict")
async def predict(file: UploadFile = File(...)) -> dict:
    try:
        img = Image.open(io.BytesIO(await file.read())).convert("RGB")
    except Exception as exc:  # noqa: BLE001
        raise HTTPException(status_code=400, detail=f"Not a readable image ({type(exc).__name__}).") from exc
    t = time.perf_counter()
    # --- verbatim from scripts/bench/cpu_inference_bench.py ---------------------
    x = proc(images=img, return_tensors="pt", do_resize=True, keep_aspect_ratio=False, size={"height": 518, "width": 518})["pixel_values"]
    # -----------------------------------------------------------------------------
    with torch.no_grad():
        depth = model(pixel_values=x.to(dev)).predicted_depth[0].float().cpu().numpy().astype("<f4")
    return {
        "model": mid,
        "kind": "relative_depth",
        "device": dev,
        "source_size": list(img.size),  # (width, height) of the input image
        "infer_s": round(time.perf_counter() - t, 3),
        **encode_depth(depth),
    }


In [ ]:
import subprocess, time, urllib.request, json
srv = subprocess.Popen(['python', '-m', 'uvicorn', 'dav2_server:app', '--host', '127.0.0.1', '--port', '8000'],
                       stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(json.load(urllib.request.urlopen('http://127.0.0.1:8000/health'))); break
    except Exception:
        assert srv.poll() is None, open('server.log').read()
        time.sleep(1)
else:
    raise RuntimeError(open('server.log').read())

In [ ]:
# cloudflared: the official static Linux binary (no account, no login for a quick tunnel)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
# Logs go to a file, not an undrained pipe (a full pipe could block cloudflared on a log write).
# Re-running this cell replaces the tunnel (NEW URL). Stopping the notebook kills it (Cloudflare error 1033).
import re
!pkill -f 'cloudflared tunnel' || true
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                       stdout=open('cloudflared.log', 'w'), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cloudflared.log').read())
    if m:
        URL = m.group(0); break
    assert tun.poll() is None, open('cloudflared.log').read()
    time.sleep(1)
print('\nDAV2_INFERENCE_URL =', URL)

In [ ]:
# Real end-to-end check through the public tunnel: a real Sentinel-2 preview -> 518x518 depth
import base64, io, numpy as np, requests
img = requests.get('https://github.com/sancharimouri/depthwizard2-assets/releases/download/library-v1/sentinel2-almora__preview.jpg', timeout=60).content
for _ in range(30):  # DNS for a fresh trycloudflare name can take a few seconds
    try:
        r = requests.post(URL + '/predict', files={'file': ('almora.jpg', img, 'image/jpeg')}, timeout=120); break
    except requests.ConnectionError:
        time.sleep(2)
r.raise_for_status(); d = r.json()
import zlib
q = np.frombuffer(zlib.decompress(base64.b64decode(d['data_b64'])), '<u2').reshape(d['shape'])  # u16-zlib
depth = d['min'] + q / 65535 * (d['max'] - d['min'])
print('payload', round(len(r.content) / 1e6, 2), 'MB (u16-zlib)')
print(d['device'], d['shape'], 'infer', d['infer_s'], 's  range', round(float(depth.min()), 3), '..', round(float(depth.max()), 3),
      ' std', round(float(depth.std()), 3))
assert depth.shape == (518, 518) and np.isfinite(depth).all() and depth.std() > 0

In [ ]:
# Keep-alive monitor: leave this running. It only reports; it cannot keep Colab alive if the tab closes.
while True:
    try:
        print(time.strftime('%H:%M:%S'), URL, 'cloudflared', 'running' if tun.poll() is None else f'EXITED {tun.returncode}',
              requests.get(URL + '/health', timeout=30).json())
    except Exception as e:
        print(time.strftime('%H:%M:%S'), 'tunnel check failed:', e)
    time.sleep(60)